# Boundary-resolved spatial transcriptional analysis

This notebook contains the analysis used for the manuscript heatmap showing the
top transcriptional changes across the granuloma–stromal boundary.

Only the final analysis workflow retained for the manuscript is included:

1. map histological-region annotations to Visium HD bins;
2. construct a signed granuloma-to-stromal distance axis;
3. aggregate log-normalized expression in spatial distance bins;
4. identify genes with significant distance-dependent expression using a
   spline-based nested-model test;
5. require a significant fitted boundary contrast and concordant direction
   across both tissue sections; and
6. display the top 20 genes enriched toward each side of the boundary as a
   smoothed heatmap.

Repository-relative paths are used in place of local-computer paths.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from patsy import dmatrix, build_design_matrices
from scipy.ndimage import gaussian_filter1d
from scipy.spatial import cKDTree
from scipy.stats import f, t
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm


## Load processed Visium HD data and histological-region annotations

The AnnData object is expected to contain:

- `obs["sample"]`
- `obs["barcode"]`
- `obs["library_id"]`
- `obsm["spatial"]`
- `uns["spatial"]`
- `layers["lognorm"]`

The region-annotation tables are expected to contain `Barcode` and `Region`
columns.


In [ ]:
DATA_PATH = Path("data/visium_hd/adata_v1_compat.h5ad")
REGION_DIR = Path("data/visium_hd/region_annotations")

REGION_FILES = {
    "09796": REGION_DIR / "Region_annot_09796.csv",
    "09811": REGION_DIR / "Region_annot_09811.csv",
}

adata = sc.read_h5ad(DATA_PATH)

required_obs = {"sample", "barcode", "library_id"}
missing_obs = required_obs.difference(adata.obs.columns)
if missing_obs:
    raise KeyError(f"Missing required obs columns: {sorted(missing_obs)}")

if "spatial" not in adata.obsm:
    raise KeyError("adata.obsm['spatial'] is required")

if "lognorm" not in adata.layers:
    raise KeyError("adata.layers['lognorm'] is required")


In [ ]:
adata.obs["histology_region"] = pd.NA

for sample, csv_path in REGION_FILES.items():
    region_table = pd.read_csv(csv_path)

    mapping = (
        region_table
        .drop_duplicates("Barcode")
        .set_index("Barcode")["Region"]
    )

    sample_mask = adata.obs["sample"].astype(str) == str(sample)

    adata.obs.loc[sample_mask, "histology_region"] = (
        adata.obs.loc[sample_mask, "barcode"]
        .astype(str)
        .map(mapping)
    )

    matched = adata.obs.loc[sample_mask, "histology_region"].notna().sum()
    print(f"{sample}: matched {matched:,}/{sample_mask.sum():,} bins")


## Construct the signed granuloma–stromal distance axis

For each tissue section, granuloma bins are assigned the negative distance to
the nearest stromal bin, whereas stromal bins are assigned the positive distance
to the nearest granuloma bin.

The Visium HD spatial coordinates are converted from full-resolution pixels to
micrometres using the 8-µm bin diameter and the corresponding library
scalefactor.


In [ ]:
GRANULOMA_REGION = "Gran"
STROMAL_REGION = "Stromal"

adata.obs["GS_signed_distance"] = np.nan

for sample in tqdm(adata.obs["sample"].astype(str).unique()):

    sample_mask = adata.obs["sample"].astype(str).to_numpy() == sample
    sample_idx = np.where(sample_mask)[0]

    region = (
        adata.obs.iloc[sample_idx]["histology_region"]
        .astype(str)
        .to_numpy()
    )

    coords = np.asarray(adata.obsm["spatial"][sample_idx])

    gran_mask = region == GRANULOMA_REGION
    stromal_mask = region == STROMAL_REGION

    print(
        f"{sample}: granuloma={gran_mask.sum():,}, "
        f"stromal={stromal_mask.sum():,}"
    )

    if gran_mask.sum() == 0 or stromal_mask.sum() == 0:
        print(f"Skipping {sample}: one region is absent")
        continue

    gran_coords = coords[gran_mask]
    stromal_coords = coords[stromal_mask]

    stromal_tree = cKDTree(stromal_coords)
    gran_dist, _ = stromal_tree.query(
        gran_coords,
        k=1,
        workers=-1,
    )

    gran_tree = cKDTree(gran_coords)
    stromal_dist, _ = gran_tree.query(
        stromal_coords,
        k=1,
        workers=-1,
    )

    signed_sample = np.full(len(sample_idx), np.nan)
    signed_sample[gran_mask] = -gran_dist
    signed_sample[stromal_mask] = stromal_dist

    adata.obs.iloc[
        sample_idx,
        adata.obs.columns.get_loc("GS_signed_distance"),
    ] = signed_sample


In [ ]:
adata.obs["GS_distance_um"] = np.nan

for sample in adata.obs["sample"].astype(str).unique():

    sample_mask = adata.obs["sample"].astype(str) == sample

    library_id = (
        adata.obs.loc[sample_mask, "library_id"]
        .astype(str)
        .iloc[0]
    )

    spot_diameter_px = (
        adata.uns["spatial"][library_id]["scalefactors"][
            "spot_diameter_fullres"
        ]
    )

    um_per_pixel = 8.0 / spot_diameter_px

    adata.obs.loc[sample_mask, "GS_distance_um"] = (
        adata.obs.loc[sample_mask, "GS_signed_distance"]
        * um_per_pixel
    )

    print(f"{sample}: {um_per_pixel:.6f} µm/pixel")


## Restrict to the displayed boundary axis and aggregate expression

The manuscript heatmap displays 300 µm on either side of the inferred boundary.
Expression is aggregated independently for each tissue section in 10-µm
distance intervals.


In [ ]:
MAX_DISTANCE_UM = 300
BIN_WIDTH_UM = 10

interface_mask = (
    adata.obs["GS_distance_um"].notna()
    & (adata.obs["GS_distance_um"].abs() <= MAX_DISTANCE_UM)
)

adata_interface = adata[interface_mask].copy()

distance_edges = np.arange(
    -MAX_DISTANCE_UM,
    MAX_DISTANCE_UM + BIN_WIDTH_UM,
    BIN_WIDTH_UM,
)

distance_centers = (
    distance_edges[:-1] + distance_edges[1:]
) / 2

adata_interface.obs["GS_distance_bin"] = pd.cut(
    adata_interface.obs["GS_distance_um"],
    bins=distance_edges,
    labels=distance_centers,
    include_lowest=True,
    right=False,
).astype(float)

print(adata_interface)


In [ ]:
all_genes = np.asarray(adata_interface.var_names)

samples = (
    adata_interface.obs["sample"]
    .astype(str)
    .unique()
)

distance_centers_observed = np.sort(
    adata_interface.obs["GS_distance_bin"]
    .dropna()
    .unique()
)

trajectory_rows = []

for sample in tqdm(samples):

    sample_mask = (
        adata_interface.obs["sample"]
        .astype(str)
        .to_numpy()
        == sample
    )

    ad_sample = adata_interface[sample_mask]

    for distance in distance_centers_observed:

        mask = (
            ad_sample.obs["GS_distance_bin"].to_numpy()
            == distance
        )

        n_bins = mask.sum()
        if n_bins == 0:
            continue

        X = ad_sample[mask].layers["lognorm"]

        mean_expr = np.asarray(
            X.mean(axis=0)
        ).ravel()

        trajectory_rows.append(
            {
                "sample": sample,
                "distance": float(distance),
                "n_bins": int(n_bins),
                "expression": mean_expr,
            }
        )

meta_df = pd.DataFrame(
    [
        {
            "sample": row["sample"],
            "distance": row["distance"],
            "n_bins": row["n_bins"],
        }
        for row in trajectory_rows
    ]
)

Y_all = np.vstack(
    [row["expression"] for row in trajectory_rows]
)

print("Trajectory matrix:", Y_all.shape)


## Filter low-expression genes

The same expression filter used for the manuscript analysis is retained.


In [ ]:
mean_expression = np.nanmean(Y_all, axis=0)
max_expression = np.nanmax(Y_all, axis=0)

gene_filter = (
    (mean_expression > 0.01)
    & (max_expression > 0.05)
)

genes_model = all_genes[gene_filter]
Y_filtered = Y_all[:, gene_filter]

print("Genes retained:", len(genes_model))


## Test for distance-dependent expression

The statistical model is fitted within ±150 µm of the boundary.

For each gene, a full model containing tissue-section covariates and a cubic
B-spline representation of signed distance is compared with a reduced model
containing the tissue-section covariates alone. Benjamini–Hochberg correction
is applied across genes.


In [ ]:
MODEL_WINDOW_UM = 150
SPLINE_DF = 5

model_mask = meta_df["distance"].abs() <= MODEL_WINDOW_UM

model_meta = (
    meta_df.loc[model_mask]
    .reset_index(drop=True)
)

Y = Y_filtered[model_mask.to_numpy(), :]

spline_formula = (
    f"bs(distance, df={SPLINE_DF}, degree=3, "
    "include_intercept=False) - 1"
)

spline_basis = dmatrix(
    spline_formula,
    data=model_meta,
    return_type="dataframe",
)

spline_info = spline_basis.design_info

sample_dummy = pd.get_dummies(
    model_meta["sample"],
    drop_first=True,
    dtype=float,
)

X_reduced = np.column_stack(
    [
        np.ones(len(model_meta)),
        sample_dummy.to_numpy(),
    ]
)

X_full = np.column_stack(
    [
        X_reduced,
        spline_basis.to_numpy(),
    ]
)

beta_reduced = np.linalg.pinv(X_reduced) @ Y
beta_full = np.linalg.pinv(X_full) @ Y

pred_reduced = X_reduced @ beta_reduced
pred_full = X_full @ beta_full

SSE_reduced = np.sum((Y - pred_reduced) ** 2, axis=0)
SSE_full = np.sum((Y - pred_full) ** 2, axis=0)

n = X_full.shape[0]
p_reduced = X_reduced.shape[1]
p_full = X_full.shape[1]

df_added = p_full - p_reduced
df_resid = n - p_full

numerator = (SSE_reduced - SSE_full) / df_added
denominator = SSE_full / df_resid

F_spatial = numerator / denominator
F_spatial = np.maximum(F_spatial, 0)

p_spatial = f.sf(
    F_spatial,
    df_added,
    df_resid,
)

q_spatial = multipletests(
    p_spatial,
    method="fdr_bh",
)[1]

spatial_results = pd.DataFrame(
    {
        "gene": genes_model,
        "F_spatial": F_spatial,
        "p_spatial": p_spatial,
        "q_spatial": q_spatial,
    }
)


## Boundary contrast

The fitted mean expression on the granuloma-proximal side (−100 to −20 µm) is
contrasted with the stromal-proximal side (+20 to +100 µm). Significance is
assessed using the fitted-model covariance and residual variance, followed by
Benjamini–Hochberg correction.


In [ ]:
left_grid = np.linspace(-100, -20, 20)
right_grid = np.linspace(20, 100, 20)

left_spline = np.asarray(
    build_design_matrices(
        [spline_info],
        {"distance": left_grid},
    )[0]
)

right_spline = np.asarray(
    build_design_matrices(
        [spline_info],
        {"distance": right_grid},
    )[0]
)

n_sample_terms = X_reduced.shape[1] - 1

X_left = np.column_stack(
    [
        np.ones(len(left_grid)),
        np.zeros((len(left_grid), n_sample_terms)),
        left_spline,
    ]
)

X_right = np.column_stack(
    [
        np.ones(len(right_grid)),
        np.zeros((len(right_grid), n_sample_terms)),
        right_spline,
    ]
)

contrast_vector = (
    X_right.mean(axis=0)
    - X_left.mean(axis=0)
)

boundary_delta = contrast_vector @ beta_full

MSE = SSE_full / df_resid

XtX_inv = np.linalg.pinv(
    X_full.T @ X_full
)

contrast_variance_factor = (
    contrast_vector
    @ XtX_inv
    @ contrast_vector.T
)

contrast_se = np.sqrt(
    MSE * contrast_variance_factor
)

t_boundary = boundary_delta / contrast_se

p_boundary = (
    2
    * t.sf(
        np.abs(t_boundary),
        df=df_resid,
    )
)

q_boundary = multipletests(
    p_boundary,
    method="fdr_bh",
)[1]

spatial_results["boundary_delta"] = boundary_delta
spatial_results["t_boundary"] = t_boundary
spatial_results["p_boundary"] = p_boundary
spatial_results["q_boundary"] = q_boundary


## Require concordant boundary direction in both tissue sections

Genes are retained when:

- the distance-dependent model has FDR < 0.05;
- the fitted boundary contrast has FDR < 0.05; and
- the observed stromal-minus-granuloma change has the same direction in each
  tissue section as the fitted boundary contrast.


In [ ]:
sample_deltas = {}

for sample in samples:

    sample_rows = (
        meta_df["sample"].astype(str)
        == sample
    )

    gran_rows = (
        sample_rows
        & meta_df["distance"].between(-100, -20)
    )

    stromal_rows = (
        sample_rows
        & meta_df["distance"].between(20, 100)
    )

    gran_mean = (
        Y_filtered[gran_rows.to_numpy()]
        .mean(axis=0)
    )

    stromal_mean = (
        Y_filtered[stromal_rows.to_numpy()]
        .mean(axis=0)
    )

    sample_deltas[sample] = (
        stromal_mean - gran_mean
    )

    spatial_results[f"delta_{sample}"] = (
        sample_deltas[sample]
    )

sample_delta_cols = [
    f"delta_{sample}"
    for sample in samples
]

model_sign = np.sign(
    spatial_results["boundary_delta"].to_numpy()
)

concordant = np.ones(
    len(spatial_results),
    dtype=bool,
)

for column in sample_delta_cols:
    concordant &= (
        np.sign(
            spatial_results[column].to_numpy()
        )
        == model_sign
    )

spatial_results["concordant"] = concordant

boundary_sig = spatial_results[
    (spatial_results["q_spatial"] < 0.05)
    & (spatial_results["q_boundary"] < 0.05)
    & spatial_results["concordant"]
].copy()

print(
    "Robust boundary-associated genes:",
    len(boundary_sig),
)


## Select the top genes in each direction

Negative boundary contrast denotes higher expression toward the granuloma side;
positive boundary contrast denotes higher expression toward the stromal side.
The manuscript heatmap displays the top 20 genes in each direction.


In [ ]:
top_granuloma = (
    boundary_sig
    .sort_values(
        "boundary_delta",
        ascending=True,
    )
    .head(20)
)

top_stromal = (
    boundary_sig
    .sort_values(
        "boundary_delta",
        ascending=False,
    )
    .head(20)
)

display(
    top_granuloma[
        [
            "gene",
            "boundary_delta",
            "q_spatial",
            "q_boundary",
            *sample_delta_cols,
        ]
    ]
)

display(
    top_stromal[
        [
            "gene",
            "boundary_delta",
            "q_spatial",
            "q_boundary",
            *sample_delta_cols,
        ]
    ]
)

gran_genes = top_granuloma["gene"].tolist()
stromal_genes = top_stromal["gene"].tolist()

gene_order = gran_genes + stromal_genes


## Generate the manuscript heatmap

For visualization, each selected gene is standardized across distance intervals
within each tissue section. Profiles are smoothed with a Gaussian kernel
(`sigma = 1.5` distance intervals) and then averaged across sections.

With the 10-µm intervals used here, `sigma = 1.5` corresponds to 15 µm along
the distance axis.


In [ ]:
gene_to_idx = {
    gene: i
    for i, gene in enumerate(all_genes)
}

selected_idx = [
    gene_to_idx[gene]
    for gene in gene_order
]

Y_selected = Y_all[:, selected_idx]

traj = meta_df[["sample", "distance"]].copy()

for i, gene in enumerate(gene_order):
    traj[gene] = Y_selected[:, i]

traj_z = traj.copy()

for sample in samples:

    mask = traj_z["sample"] == sample

    for gene in gene_order:

        values = (
            traj_z.loc[mask, gene]
            .to_numpy(dtype=float)
        )

        mu = np.nanmean(values)
        sd = np.nanstd(values)

        if sd > 0:
            traj_z.loc[mask, gene] = (
                values - mu
            ) / sd
        else:
            traj_z.loc[mask, gene] = 0

SMOOTH_SIGMA = 1.5

for sample in samples:

    mask = traj_z["sample"] == sample

    ordered_idx = (
        traj_z.loc[mask]
        .sort_values("distance")
        .index
    )

    for gene in gene_order:
        traj_z.loc[
            ordered_idx,
            gene + "_smooth",
        ] = gaussian_filter1d(
            traj_z.loc[
                ordered_idx,
                gene,
            ].to_numpy(),
            sigma=SMOOTH_SIGMA,
        )

smooth_cols = [
    gene + "_smooth"
    for gene in gene_order
]

trajectory_mean = (
    traj_z
    .groupby("distance")[smooth_cols]
    .mean()
    .sort_index()
)

trajectory_mean.columns = [
    column.replace("_smooth", "")
    for column in trajectory_mean.columns
]

heatmap_data = (
    trajectory_mean[gene_order]
    .T
)


In [ ]:
fig, ax = plt.subplots(
    figsize=(16, 11),
    dpi=300,
)

plot_data = heatmap_data.clip(-2, 2)

im = ax.imshow(
    plot_data.to_numpy(),
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    origin="upper",
    vmin=-2,
    vmax=2,
    extent=[
        -300,
        300,
        len(plot_data.index) - 0.5,
        -0.5,
    ],
)

ax.set_yticks(
    np.arange(len(plot_data.index))
)

ax.set_yticklabels(
    plot_data.index,
    fontsize=9,
)

ax.set_xticks(
    np.arange(-300, 301, 50)
)

ax.tick_params(
    axis="x",
    length=0,
)

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

ax.axhline(
    len(gran_genes) - 0.5,
    color="black",
    linewidth=0.5,
)

ax.set_xlabel(
    "Distance from granuloma–stromal boundary (µm)"
)

ax.set_ylabel(
    "Spatial boundary-associated genes"
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.025,
    pad=0.03,
)

cbar.set_label(
    "Relative expression (z-score)",
    rotation=270,
    labelpad=18,
)

plt.tight_layout()
plt.show()
